# Differential Accessibility (DESeq2 on consensus peaks) - headless

Reproduces a paper's differential-peak finding from a consensus-peak count matrix.


In [ ]:
# Parameters (injected at launch). All are str/number so injection stays valid R.
counts_path <- "/data/consensus_peaks.featureCounts.txt"   # featureCounts over consensus peaks
output_path <- "/outputs/da_results.csv"
test_samples <- ""                            # comma-separated sample names (treatment)
reference_samples <- ""                       # comma-separated sample names (control)
block_labels <- ""                            # optional matched-pairs: per-sample block/subject
                                              # labels, comma-separated, ALIGNED to c(test,reference)
lfc_threshold <- 1.0
padj_threshold <- 0.05


In [ ]:
suppressMessages(library(DESeq2))

test_s <- trimws(strsplit(test_samples, ",")[[1]]); test_s <- test_s[test_s != ""]
ref_s  <- trimws(strsplit(reference_samples, ",")[[1]]); ref_s <- ref_s[ref_s != ""]
stopifnot(length(test_s) > 0, length(ref_s) > 0)
samples <- c(test_s, ref_s)
condition <- factor(c(rep("test", length(test_s)), rep("reference", length(ref_s))), levels = c("reference", "test"))

# featureCounts writes a leading '# Program...' comment line; comment.char skips it.
mat <- read.delim(counts_path, check.names = FALSE, comment.char = '#', stringsAsFactors = FALSE)
coord <- c("Chr", "Start", "End")
if (!all(coord %in% colnames(mat))) stop("expected featureCounts Chr/Start/End columns")

# A declared sample is an accession (SRX9040493); nf-core names the matrix column after the merged
# BAM it counted (SRX9040493_REP1.mLb.clN.sorted.bam). Exact matching therefore matched NOTHING and
# every atacseq/chipseq Level-3 run aborted here. Resolve by accession-prefix, and refuse on an
# ambiguous or absent match rather than silently analysing the wrong column.
resolve_col <- function(s, cols) {
  exact <- cols[cols == s]
  if (length(exact) == 1) return(exact)
  hits <- cols[startsWith(cols, paste0(s, "_")) | startsWith(cols, paste0(s, "."))]
  if (length(hits) == 1) return(hits)
  if (length(hits) == 0) {
    stop(paste0("sample '", s, "' matches no column in the counts matrix; columns: ",
                paste(utils::head(cols, 20), collapse = ", ")))
  }
  stop(paste0("sample '", s, "' matches ", length(hits), " columns (", paste(hits, collapse = ", "),
              "); cannot choose one"))
}
sample_cols <- vapply(samples, function(s) resolve_col(s, colnames(mat)), character(1), USE.NAMES = FALSE)
coldata <- data.frame(condition = condition, row.names = sample_cols)

# Matched-pairs / blocked design, mirroring de_bulk_deseq2: when a block label is supplied for EVERY
# sample and there are >= 2 distinct labels, model `~ block + condition` so donor-to-donor baseline
# variance is removed. This notebook previously ignored block_labels entirely, so a plan that
# declared a paired design (and passed validate_paired_designs) was analysed unpaired without saying so.
if (!exists("block_labels")) block_labels <- ""   # tolerate an injector/DB row that omits this optional param
block_s <- trimws(strsplit(block_labels, ",")[[1]]); block_s <- block_s[block_s != ""]
use_block <- length(block_s) == length(samples) && length(unique(block_s)) >= 2
if (use_block) {
  coldata$block <- factor(block_s)
  design_formula <- ~ block + condition
} else {
  design_formula <- ~ condition
}

counts <- as.matrix(mat[, sample_cols, drop = FALSE])
counts <- matrix(as.integer(round(as.numeric(counts))), nrow = nrow(counts), dimnames = dimnames(counts))
rownames(counts) <- make.unique(as.character(mat$Geneid))

dds <- DESeqDataSetFromMatrix(countData = counts, colData = coldata, design = design_formula)
dds <- DESeq(dds)
res <- as.data.frame(results(dds, contrast = c("condition", "test", "reference")))
dir.create(dirname(output_path), showWarnings = FALSE, recursive = TRUE)
out <- data.frame(chr = mat$Chr, start = mat$Start, end = mat$End,
                  log2FoldChange = res$log2FoldChange, padj = res$padj)
write.csv(out, output_path, row.names = FALSE)
cat("wrote", nrow(out), "peaks to", output_path, "\n")
if (use_block) cat("design: ~ block + condition (paired,", length(unique(block_s)), "subjects)\n")
